# 06 · Extraer, diagnosticar y segmentar textos completos

Esta fase transforma los PDF locales registrados en la fase 05 en un corpus auditable por página, sección y fragmento. No realiza OCR ni consultas de red.

Flujo:

```text
full_text_manifest.csv
    → validación de archivo y checksum
    → extracción por página
    → diagnóstico de páginas y documentos
    → detección conservadora de secciones
    → fragmentos con rango de páginas
    → corpus para cribado a texto completo
```


## Principios

- El PDF original nunca se modifica.
- Cada salida conserva `source_id`, ruta, checksum y páginas.
- Los errores de una página no detienen el resto del corpus.
- Los documentos probablemente escaneados se marcan para OCR, pero no se procesan automáticamente.
- La detección de secciones es una ayuda de navegación y debe validarse antes de usarla como evidencia.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from evidence_review.document_parsing import (
    load_document_parsing_config,
    parsing_summary,
    read_manifest_csv,
    run_document_parsing,
)

CONFIG_PATH = ROOT / "config" / "document_parsing.yml"
config = load_document_parsing_config(CONFIG_PATH)

print(f"Project root: {ROOT}")
print(f"Config: {CONFIG_PATH.relative_to(ROOT)}")


## 1. Cargar el manifiesto local

La entrada contiene únicamente archivos registrados como `available_local` en la fase 05. Con los resultados actuales se esperan 16 documentos.


In [ ]:
manifest_path = ROOT / config["paths"]["manifest_csv"]

if not manifest_path.exists():
    raise FileNotFoundError(
        f"No existe {manifest_path}. Ejecuta primero la fase 05 y exporta "
        "full_text_manifest.csv."
    )

manifest, manifest_encoding = read_manifest_csv(manifest_path)

print(f"Manifest encoding: {manifest_encoding}")
print(f"Documents in manifest: {len(manifest)}")
print(f"Unique source_id: {manifest['source_id'].nunique()}")

display(
    manifest[
        [
            column
            for column in (
                "source_id",
                "title",
                "screening_decision",
                "local_path",
                "file_size_bytes",
                "checksum_sha256",
            )
            if column in manifest.columns
        ]
    ].head(20)
)


## 2. Ejecutar extracción y segmentación

La ejecución es local. Verifica firma PDF y checksum, extrae texto con `pypdf`, registra páginas vacías o con errores, identifica posibles documentos escaneados y genera fragmentos solapados.


In [ ]:
RUN_PARSING = True

if not RUN_PARSING:
    raise RuntimeError("Set RUN_PARSING = True to execute the local parsing phase.")

results = run_document_parsing(
    manifest,
    config,
    project_root=ROOT,
)

inventory = results["inventory"]
page_text = results["page_text"]
page_diagnostics = results["page_diagnostics"]
sections = results["sections"]
chunks = results["chunks"]
parsing_errors = results["errors"]
parsing_issues = results["issues"]
ocr_queue = results["ocr_queue"]
screening_corpus = results["screening_corpus"]

display(
    parsing_summary(
        inventory,
        page_text,
        sections,
        chunks,
        parsing_errors,
    )
)


## 3. Revisar inventario y diagnóstico

Presta atención a:

- `parsing_status = failed`;
- `checksum_matches = false`;
- `pages_error > 0`;
- `ocr_recommended = true`;
- documentos con muy pocos caracteres extraídos.


In [ ]:
display(
    inventory[
        [
            "source_id",
            "title",
            "page_count",
            "pages_with_text",
            "pages_low_text",
            "pages_empty",
            "pages_error",
            "total_characters",
            "low_or_empty_fraction",
            "ocr_recommended",
            "parsing_status",
        ]
    ].sort_values(
        ["ocr_recommended", "pages_error", "total_characters"],
        ascending=[False, False, True],
    )
)

print(f"Documents requiring OCR review: {len(ocr_queue)}")
display(ocr_queue.head(50))

print(f"Parsing errors: {len(parsing_errors)}")
display(parsing_errors.head(50))


## 4. Examinar páginas, secciones y fragmentos

Los rangos de páginas permiten regresar al PDF original. La etiqueta de sección es heurística y no reemplaza la lectura humana.


In [ ]:
display(
    page_diagnostics[
        [
            "source_id",
            "page_number",
            "extraction_status",
            "character_count",
            "word_count",
            "possible_scanned_page",
            "error_type",
        ]
    ].head(50)
)

print(f"Sections: {len(sections)}")
display(
    sections[
        [
            "section_id",
            "source_id",
            "section_heading",
            "heading_detection_method",
            "start_page",
            "end_page",
            "character_count",
        ]
    ].head(50)
)

print(f"Chunks: {len(chunks)}")
display(
    chunks[
        [
            "chunk_id",
            "source_id",
            "section_heading",
            "chunk_index",
            "start_page",
            "end_page",
            "character_count",
            "text",
        ]
    ].head(20)
)


## 5. Validar consistencia cruzada

La validación comprueba identificadores duplicados, conteos de páginas, checksum, rangos de páginas y fragmentos vacíos.


In [ ]:
print(f"Validation issues: {len(parsing_issues)}")
display(parsing_issues.head(100))

if not parsing_issues.empty:
    print(
        "La fase puede exportarse para auditoría, pero no debe considerarse "
        "cerrada hasta revisar los problemas mostrados."
    )


## 6. Exportar inventario, corpus y auditoría

Las salidas se escriben en `data/interim/`; los textos completos con marcadores de página se escriben en `data/processed/text/`. Ambos directorios permanecen fuera de Git.


In [ ]:
output_frames = {
    config["paths"]["inventory_csv"]: inventory,
    config["paths"]["page_text_csv"]: page_text,
    config["paths"]["page_diagnostics_csv"]: page_diagnostics,
    config["paths"]["sections_csv"]: sections,
    config["paths"]["chunks_csv"]: chunks,
    config["paths"]["errors_csv"]: parsing_errors,
    config["paths"]["issues_csv"]: parsing_issues,
    config["paths"]["ocr_queue_csv"]: ocr_queue,
    config["paths"]["screening_corpus_csv"]: screening_corpus,
}

for relative_path, frame in output_frames.items():
    output_path = ROOT / relative_path
    output_path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"{output_path.relative_to(ROOT)}: {len(frame)}")


## 7. Resumen final y criterio para avanzar

La fase 06 está lista para cerrarse cuando:

1. todos los documentos del manifiesto tienen un estado explícito;
2. no existen errores de validación estructural;
3. cada documento parseado conserva checksum y páginas;
4. los documentos escaneados están en la cola de OCR;
5. los fragmentos tienen `source_id`, sección y rango de páginas;
6. las fuentes `uncertain` disponibles están listas para cribado a texto completo.


In [ ]:
summary = parsing_summary(
    inventory,
    page_text,
    sections,
    chunks,
    parsing_errors,
)
display(summary)

print(
    "Uncertain sources with local full text:",
    int(screening_corpus["full_text_screening_required"].eq("true").sum()),
)
print("Validation issues:", len(parsing_issues))
print("Next phase: 07_full_text_screening.ipynb")


## Siguiente fase

```text
notebooks/07_full_text_screening.ipynb
```

La fase 07 aplicará los criterios de elegibilidad al texto completo, resolverá las fuentes `uncertain`, registrará razones de exclusión y definirá el corpus final que pasará a extracción estructurada de hallazgos.
